In [1]:
# Cell 0 — install & explore
!pip install datasketch pyarrow -q

import os
import pyarrow.dataset as ds
import pyarrow.parquet as pq

DATA_DIR = "/kaggle/input/datasets/datdong123/amazon-clean-v2"

# --- xem có bao nhiêu file và tổng dung lượng ---
files = []
for root, dirs, fnames in os.walk(DATA_DIR):
    for f in fnames:
        if f.endswith(".parquet"):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / 1024**2
            files.append((full, size_mb))
            
print(f"Tổng số file: {len(files)}")
print(f"Tổng dung lượng: {sum(s for _, s in files):.1f} MB")
for path, mb in files[:5]:
    print(f"  {os.path.basename(path):40s}  {mb:7.1f} MB")

# --- xem schema của file đầu tiên (không load data) ---
sample_file = files[0][0]
schema = pq.read_schema(sample_file)
print("\nSchema:")
print(schema)

# --- đọc vài dòng để xem dữ liệu thực tế ---
sample = pq.read_table(sample_file).to_pandas().head(3)
print("\nSample rows:")
print(sample)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.5 MB/s eta 0:00:00
Tổng số file: 200
Tổng dung lượng: 9577.8 MB
  part-00129-aa4c9f9a-ce05-426c-99e6-eb82cc252ad7-c000.snappy.parquet     48.0 MB
  part-00127-aa4c9f9a-ce05-426c-99e6-eb82cc252ad7-c000.snappy.parquet     48.0 MB
  part-00193-aa4c9f9a-ce05-426c-99e6-eb82cc252ad7-c000.snappy.parquet     47.9 MB
  part-00099-aa4c9f9a-ce05-426c-99e6-eb82cc252ad7-c000.snappy.parquet     47.7 MB
  part-00042-aa4c9f9a-ce05-426c-99e6-eb82cc252ad7-c000.snappy.parquet     48.2 MB

Schema:
product_id: string
customer_id: string
review_id: string
product_parent: string
product_title: string
star_rating: int32
helpful_votes: int32
total_votes: int32
vine: int32
verified_purchase: int32
review_headline: string
review_body: string
review_date: date32[day]
helpful_ratio: double
review_length: int32
product_category: string
-- schema metadata --
org.apache.spark.version: '4.0.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","field

In [2]:
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd

# Dùng dataset API — lazy, chỉ scan metadata
dataset = ds.dataset(DATA_DIR, format="parquet")

print(f"Tổng columns: {dataset.schema.names}")

# Đếm rows mà không load data vào RAM
total_rows = dataset.count_rows()
print(f"Tổng số reviews: {total_rows:,}")

# Lấy mẫu nhỏ để EDA — stratified theo file
sample_df = dataset.head(200_000).to_pandas()

print(f"\n--- Rating distribution ---")
print(sample_df["star_rating"].value_counts().sort_index())

print(f"\n--- Top categories ---")
print(sample_df["product_category"].value_counts().head(20))

print(f"\n--- Review length stats ---")
print(sample_df["review_length"].describe(percentiles=[.25,.5,.75,.95,.99]))

print(f"\n--- Helpful ratio (chỉ reviews có >= 5 votes) ---")
has_votes = sample_df[sample_df["total_votes"] >= 5]
print(f"  Reviews có votes: {len(has_votes):,} ({len(has_votes)/len(sample_df)*100:.1f}%)")
print(has_votes["helpful_ratio"].describe())

print(f"\n--- Null check ---")
print(sample_df[["review_body","review_headline","star_rating","helpful_ratio"]].isnull().sum())

print(f"\n--- review_body rỗng (empty string) ---")
empty_body = (sample_df["review_body"].str.strip() == "").sum()
print(f"  Empty review_body: {empty_body:,}")

print(f"\n--- Verified purchase ratio ---")
print(sample_df["verified_purchase"].value_counts(normalize=True).round(3))

Tổng columns: ['product_id', 'customer_id', 'review_id', 'product_parent', 'product_title', 'star_rating', 'helpful_votes', 'total_votes', 'vine', 'verified_purchase', 'review_headline', 'review_body', 'review_date', 'helpful_ratio', 'review_length', 'product_category']
Tổng số reviews: 31,862,547

--- Rating distribution ---
star_rating
1     12759
2      9217
3     16811
4     36078
5    125135
Name: count, dtype: int64

--- Top categories ---
product_category
PC                        16558
Wireless                  16402
Video DVD                 16120
Digital_Ebook_Purchase    11954
Mobile_Apps               11065
Electronics                9599
Pet Products               9397
Digital_Video_Download     9167
Sports                     8860
Health & Personal Care     8280
Office Products            7842
Beauty                     7553
Grocery                    7485
Toys                       7093
Automotive                 6331
Video Games                6039
Tools                

In [3]:
# Cell 2 — SimHash dedup (numpy vectorized)
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import re, gc, os, time
from hashlib import md5

OUT_CLEAN = "/kaggle/working/clean"
os.makedirs(OUT_CLEAN, exist_ok=True)

COLS = [
    "review_id", "product_id", "star_rating",
    "helpful_votes", "total_votes", "helpful_ratio",
    "verified_purchase", "review_headline", "review_body",
    "product_category", "review_date",
]

N_BITS   = 64
HAM_DIST = 3

# Lookup table: mỗi byte → 8 bits (+1/-1), shape (256, 8)
_BITS = np.array([
    [(1 if (b >> i) & 1 else -1) for i in range(8)]
    for b in range(256)
], dtype=np.int16)

def simhash_batch(texts: list[str]) -> np.ndarray:
    results = np.zeros(len(texts), dtype=np.uint64)

    for idx, text in enumerate(texts):
        text = re.sub(r'\s+', ' ', text.lower().strip())
        if len(text) < 20:
            continue

        words = text.split()
        if not words:
            continue

        # Hash mỗi word → lấy 8 bytes đầu của md5
        raw = b''.join(md5(w.encode()).digest()[:8] for w in words)
        hashes = np.frombuffer(raw, dtype=np.uint8).reshape(len(words), 8)
        # (n_words, 8 bytes) → (n_words, 8 bytes, 8 bits) → sum → (8, 8) → flatten → (64,)
        v = _BITS[hashes].sum(axis=0).flatten()   # shape: (64,)

        # Pack 64 bits thành uint64
        bits = np.packbits((v > 0).astype(np.uint8), bitorder='little')  # 8 bytes
        results[idx] = bits.view(np.uint64)[0]

    return results

# ── Pass 1: tính simhash ───────────────────────────────────────────────────────
print("Pass 1: computing SimHash...")
t0        = time.time()
all_sigs  = []   # list of DataFrames nhỏ
BATCH     = 10_000

for file_idx, (fpath, _) in enumerate(files):
    df = pq.read_table(
        fpath,
        columns=["review_id", "review_headline", "review_body", "product_category"]
    ).to_pandas()

    texts = (
        df["review_headline"].fillna("") + " " + df["review_body"].fillna("")
    ).str.strip().tolist()

    # Process theo batch
    sh_vals = np.zeros(len(df), dtype=np.uint64)
    for i in range(0, len(df), BATCH):
        sh_vals[i:i+BATCH] = simhash_batch(texts[i:i+BATCH])

    all_sigs.append(pd.DataFrame({
        "review_id": df["review_id"].values,
        "category":  df["product_category"].values,
        "sh":        sh_vals,
    }))

    del df, texts, sh_vals; gc.collect()

    if (file_idx + 1) % 5 == 0:
        elapsed = time.time() - t0
        eta_min = (elapsed / (file_idx+1)) * (len(files) - file_idx - 1) / 60
        n_rows  = sum(len(s) for s in all_sigs)
        print(
            f"  [{file_idx+1:3d}/200]  "
            f"rows={n_rows:,}  "
            f"{elapsed:.0f}s  ETA={eta_min:.0f}m"
        )

sig_df = pd.concat(all_sigs, ignore_index=True)
del all_sigs; gc.collect()
print(f"\nSig DataFrame: {len(sig_df):,} rows | "
      f"{sig_df.memory_usage(deep=True).sum()/1024**3:.2f} GB")

# ── Pass 2: dedup per category (sliding window) ───────────────────────────────
print("\nPass 2: finding duplicates...")
dup_ids = set()

for cat in sorted(sig_df["category"].unique()):
    cat_df = sig_df[sig_df["category"] == cat].sort_values("sh").reset_index(drop=True)
    sh_arr = cat_df["sh"].to_numpy(dtype=np.uint64)
    id_arr = cat_df["review_id"].to_numpy()
    n      = len(cat_df)

    # Vectorize hamming distance — so sánh tất cả cặp liền kề cùng lúc
    xor    = sh_arr[1:].astype(np.int64) ^ sh_arr[:-1].astype(np.int64)
    # Đếm số bit 1 trong xor bằng numpy (popcount)
    hd     = np.array([bin(int(x)).count('1') for x in xor])
    is_dup = hd <= HAM_DIST

    dup_ids.update(id_arr[1:][is_dup])
    cat_dups = is_dup.sum()
    print(f"  {cat:35s}  n={n:7,}  dups={cat_dups:5,}  rate={cat_dups/n*100:.1f}%")

    del cat_df; gc.collect()

del sig_df; gc.collect()
print(f"\nTổng duplicates: {len(dup_ids):,}")

# ── Pass 3: filter và ghi clean files ─────────────────────────────────────────
print("\nPass 3: writing clean files...")
t0      = time.time()
FLUSH   = 300_000
buffers = {}

def flush(cat: str):
    if not buffers.get(cat):
        return
    slug     = cat.replace(' ', '_').replace('/', '_')
    out_path = os.path.join(OUT_CLEAN, f"{slug}.parquet")
    df_out   = pd.concat(buffers[cat], ignore_index=True)
    if os.path.exists(out_path):
        df_out = pd.concat([pd.read_parquet(out_path), df_out], ignore_index=True)
    df_out.to_parquet(out_path, index=False, compression="snappy")
    buffers[cat] = []
    del df_out; gc.collect()

for file_idx, (fpath, _) in enumerate(files):
    df = pq.read_table(fpath, columns=COLS).to_pandas()

    for cat, grp in df.groupby("product_category"):
        clean = grp[~grp["review_id"].isin(dup_ids)]
        if len(clean):
            buffers.setdefault(cat, []).append(clean)
        if sum(len(b) for b in buffers.get(cat, [])) >= FLUSH:
            flush(cat)

    del df; gc.collect()

    if (file_idx + 1) % 20 == 0:
        elapsed = time.time() - t0
        eta_min = (elapsed / (file_idx+1)) * (len(files) - file_idx - 1) / 60
        print(f"  [{file_idx+1:3d}/200]  {elapsed:.0f}s  ETA={eta_min:.0f}m")

for cat in list(buffers.keys()):
    flush(cat)

# ── Summary ───────────────────────────────────────────────────────────────────
clean_files = [f for f in os.listdir(OUT_CLEAN) if f.endswith(".parquet")]
total_size  = sum(os.path.getsize(os.path.join(OUT_CLEAN, f)) for f in clean_files)
print(f"\n{'='*60}")
print(f"Duplicates removed: {len(dup_ids):,}")
print(f"Output: {len(clean_files)} files | {total_size/1024**3:.2f} GB")

Pass 1: computing SimHash...
  [  5/200]  rows=797,612  110s  ETA=71m
  [ 10/200]  rows=1,594,038  219s  ETA=69m
  [ 15/200]  rows=2,389,151  328s  ETA=67m
  [ 20/200]  rows=3,186,524  438s  ETA=66m
  [ 25/200]  rows=3,982,998  547s  ETA=64m
  [ 30/200]  rows=4,778,762  656s  ETA=62m
  [ 35/200]  rows=5,575,324  766s  ETA=60m
  [ 40/200]  rows=6,374,665  877s  ETA=58m
  [ 45/200]  rows=7,170,243  986s  ETA=57m
  [ 50/200]  rows=7,968,142  1096s  ETA=55m
  [ 55/200]  rows=8,763,156  1206s  ETA=53m
  [ 60/200]  rows=9,558,589  1315s  ETA=51m
  [ 65/200]  rows=10,354,945  1425s  ETA=49m
  [ 70/200]  rows=11,152,757  1535s  ETA=48m
  [ 75/200]  rows=11,948,543  1645s  ETA=46m
  [ 80/200]  rows=12,743,929  1754s  ETA=44m
  [ 85/200]  rows=13,539,127  1864s  ETA=42m
  [ 90/200]  rows=14,336,662  1974s  ETA=40m
  [ 95/200]  rows=15,133,464  2083s  ETA=38m
  [100/200]  rows=15,929,929  2193s  ETA=37m
  [105/200]  rows=16,727,822  2304s  ETA=35m
  [110/200]  rows=17,524,197  2413s  ETA=33m
  [1

In [4]:
import os

# 1. Điền tên đăng nhập Kaggle của bạn (chữ thường, viết liền, không dấu)
# Ví dụ link profile của bạn là kaggle.com/nguyenvana thì điền "nguyenvana"
os.environ['KAGGLE_USERNAME'] = "nlvdull" 

# 2. Dán nguyên cái chuỗi Token bạn vừa copy vào đây
os.environ['KAGGLE_KEY'] = "KGAT_2a3e48c9252514cf84192c5c52a4973e" 

print("Đã nạp xong Token!")

Đã nạp xong Token!


In [5]:
!kaggle datasets init -p /kaggle/working/clean

Data package template written to: /kaggle/working/clean/dataset-metadata.json


In [6]:
import json

metadata = {
  "title": "Cleaned Reviews Data",
  "id": "nlvdull/cleaned-reviews-data", 
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ]
}

# 2. Ghi đè nội dung này vào file metadata trong thư mục clean
file_path = '/kaggle/working/clean/dataset-metadata.json'
with open(file_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print("Đã cập nhật file metadata thành công!")

Đã cập nhật file metadata thành công!


In [7]:
!kaggle datasets create -p /kaggle/working/clean

Starting upload for file Digital_Ebook_Purchase.parquet
100%|█████████████████████████████████████████| 648M/648M [00:05<00:00, 126MB/s]
Upload successful: Digital_Ebook_Purchase.parquet (648MB)
Starting upload for file Video_DVD.parquet
100%|█████████████████████████████████████████| 817M/817M [00:06<00:00, 127MB/s]
Upload successful: Video_DVD.parquet (817MB)
Starting upload for file Pet_Products.parquet
100%|█████████████████████████████████████████| 277M/277M [00:02<00:00, 109MB/s]
Upload successful: Pet_Products.parquet (277MB)
Starting upload for file Camera.parquet
100%|█████████████████████████████████████████| 197M/197M [00:01<00:00, 129MB/s]
Upload successful: Camera.parquet (197MB)
Starting upload for file Grocery.parquet
100%|█████████████████████████████████████████| 192M/192M [00:01<00:00, 133MB/s]
Upload successful: Grocery.parquet (192MB)
Starting upload for file Outdoors.parquet
100%|█████████████████████████████████████████| 179M/179M [00:01<00:00, 110MB/s]
Upload suc